In [1]:
# ============================================================================
# AIRLINE LOYALTY — CHURN PREDICTION PIPELINE (end-to-end, runnable)
# ============================================================================
# Continues the agreed project decisions:
#   - Cleaning: dedupe cfa (drop exact copies, then sum partial-month rows),
#     abs() negative salaries, drop constant Country, NaN cancellation = active.
#   - Prediction point T = end of June 2018. Features use ONLY data <= T.
#   - Population: flew at least once in trailing 12m (Jul'17-Jun'18) AND not
#     cancelled by T.
#   - Churn label: no flights in H2 2018 (Jul-Dec) OR cancelled in H2 2018.
#   - CLV and Salary excluded from the model (leakage / missingness); CLV is
#     used ONLY downstream for the value axis of segmentation.
# Outputs:
#   - final_customer_results.csv  (scored customers + segment + action)
#   - validation prints throughout (CV AUC, threshold sweep, confusion
#     matrices, lift/capture, feature importance).
# ============================================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, average_precision_score,
                             precision_recall_curve, precision_score,
                             recall_score, f1_score)
from xgboost import XGBClassifier

pd.set_option('display.width', 160)
RNG = 0

# ----------------------------------------------------------------------------
# PHASE 1-2 — LOAD + CLEAN
# ----------------------------------------------------------------------------
 # <-- change to '' on your machine
clh = pd.read_csv('Customer_Loyalty_History.csv')
cfa = pd.read_csv('Customer_Flight_Activity.csv')

print('=' * 70)
print('PHASE 1-2: CLEANING')
print('=' * 70)
print(f'Raw cfa rows: {len(cfa):,} | raw clh rows: {len(clh):,}')

keys = ['Loyalty Number', 'Year', 'Month']
n_dup_keys = cfa.duplicated(subset=keys).sum()
n_dup_full = cfa.duplicated().sum()
print(f'Duplicate (cust,year,month) keys: {n_dup_keys:,} '
      f'(exact copies: {n_dup_full:,}, conflicting: {n_dup_keys - n_dup_full:,})')

cfa_cleaned = (cfa.drop_duplicates()                 # drop exact copies
                  .groupby(keys, as_index=False)     # sum partial-month rows
                  .sum())
print(f'cfa_cleaned rows: {len(cfa_cleaned):,} (expected 389,065)')

clh_cleaned = clh.copy()
n_neg = (clh_cleaned['Salary'] < 0).sum()
clh_cleaned['Salary'] = clh_cleaned['Salary'].abs()
clh_cleaned = clh_cleaned.drop(columns=['Country'])  # constant
print(f'Negative salaries fixed: {n_neg} | Country column dropped (constant)')
print(f'clh_cleaned rows: {len(clh_cleaned):,}')

# month index that crosses year boundaries cleanly
cfa_cleaned['ym'] = cfa_cleaned['Year'] * 12 + cfa_cleaned['Month']
clh_cleaned['cancel_ym'] = (clh_cleaned['Cancellation Year'] * 12
                            + clh_cleaned['Cancellation Month'])

# ----------------------------------------------------------------------------
# PHASE 4 — CHURN DEFINITION (prediction point T = June 2018)
# ----------------------------------------------------------------------------
T = 2018 * 12 + 6
END_2018 = 2018 * 12 + 12

print('\n' + '=' * 70)
print('PHASE 4: CHURN DEFINITION  (T = end of June 2018)')
print('=' * 70)

recent12 = cfa_cleaned[(cfa_cleaned['ym'] >= T - 11) & (cfa_cleaned['ym'] <= T)]
recent_flights = recent12.groupby('Loyalty Number')['Total Flights'].sum()
active_recent = recent_flights[recent_flights > 0]
cancelled_by_T = set(clh_cleaned.loc[clh_cleaned['cancel_ym'] <= T, 'Loyalty Number'])
population_ids = set(active_recent.index) - cancelled_by_T

h2 = cfa_cleaned[(cfa_cleaned['ym'] > T) & (cfa_cleaned['ym'] <= END_2018)]
h2_flights = h2.groupby('Loyalty Number')['Total Flights'].sum()
flew_h2 = set(h2_flights[h2_flights > 0].index)
cancelled_h2 = set(clh_cleaned.loc[(clh_cleaned['cancel_ym'] > T) &
                                   (clh_cleaned['cancel_ym'] <= END_2018),
                                   'Loyalty Number'])

churned_ids = (population_ids - flew_h2) | (population_ids & cancelled_h2)

customers = pd.DataFrame(index=pd.Index(sorted(population_ids),
                                        name='Loyalty Number'))
customers['churned'] = customers.index.isin(churned_ids).astype(int)

print(f'Population (active Jul17-Jun18, not cancelled by T): {len(customers):,}')
print(f'Churners: {customers["churned"].sum():,} '
      f'({customers["churned"].mean()*100:.1f}%)')

# ----------------------------------------------------------------------------
# PHASE 5 — FEATURE ENGINEERING (everything from ym <= T only)
# ----------------------------------------------------------------------------
print('\n' + '=' * 70)
print('PHASE 5: FEATURE ENGINEERING')
print('=' * 70)

hist = cfa_cleaned[cfa_cleaned['ym'] <= T]
active = hist[hist['Total Flights'] > 0]

def window(df, n):
    """rows in the last n months ending at T"""
    return df[(df['ym'] >= T - n + 1) & (df['ym'] <= T)]

def agg(df, col, how='sum'):
    return getattr(df.groupby('Loyalty Number')[col], how)().reindex(customers.index).fillna(0)

w3, w6, w12 = window(hist, 3), window(hist, 6), window(hist, 12)

# --- recency ---
last_active = active.groupby('Loyalty Number')['ym'].max()
customers['recency_months'] = (T - last_active).reindex(customers.index).fillna(99)

# --- frequency at multiple horizons (captures recent deceleration) ---
customers['flights_3m']  = agg(w3,  'Total Flights')
customers['flights_6m']  = agg(w6,  'Total Flights')
customers['flights_12m'] = agg(w12, 'Total Flights')

act3  = w3[w3['Total Flights'] > 0]
act6  = w6[w6['Total Flights'] > 0]
act12 = w12[w12['Total Flights'] > 0]
customers['active_months_3m']  = agg(act3,  'Total Flights', 'count')
customers['active_months_6m']  = agg(act6,  'Total Flights', 'count')
customers['active_months_12m'] = agg(act12, 'Total Flights', 'count')

# --- momentum / trend ---
prior6 = hist[(hist['ym'] >= T - 11) & (hist['ym'] <= T - 6)]   # Jul-Dec 2017
customers['flights_prior6'] = agg(prior6, 'Total Flights')
customers['trend_6v6'] = customers['flights_6m'] - customers['flights_prior6']
customers['ratio_6v6'] = (customers['flights_6m'] + 1) / (customers['flights_prior6'] + 1)
# share of 12m activity that happened in the last 3 months (0 = front-loaded, fading)
customers['share_last3_of_12'] = customers['flights_3m'] / (customers['flights_12m'] + 1)

# --- recency-weighted activity: recent months count more ---
w = w12.copy()
w['wgt'] = 0.8 ** (T - w['ym'])                       # decay: this month=1, 11 mo ago=0.086
w['wflights'] = w['wgt'] * w['Total Flights']
customers['flights_decayed'] = agg(w, 'wflights')

# --- distance & intensity ---
customers['distance_12m'] = agg(w12, 'Distance')
customers['distance_per_flight'] = customers['distance_12m'] / (customers['flights_12m'] + 1)
customers['avg_flights_per_active_month'] = (
    customers['flights_12m'] / customers['active_months_12m'].replace(0, np.nan)).fillna(0)

# --- points engagement ---
customers['points_acc_12m'] = agg(w12, 'Points Accumulated')
customers['points_red_12m'] = agg(w12, 'Points Redeemed')
customers['redemption_ratio'] = (customers['points_red_12m']
                                 / (customers['points_acc_12m'] + 1))
customers['ever_redeemed_12m'] = (customers['points_red_12m'] > 0).astype(int)
red6 = agg(w6, 'Points Redeemed')
customers['redeemed_6m_flag'] = (red6 > 0).astype(int)

# --- regularity: gaps and volatility over a proper ym pivot (trailing 12m) ---
piv = w12.pivot_table(index='Loyalty Number', columns='ym',
                      values='Total Flights', aggfunc='sum', fill_value=0)
piv = piv.reindex(columns=range(T - 11, T + 1), fill_value=0).reindex(customers.index, fill_value=0)

def longest_zero_run_and_tail(row):
    longest = run = tail = 0
    for v in row:
        run = run + 1 if v == 0 else 0
        longest = max(longest, run)
    for v in row[::-1]:
        if v == 0: tail += 1
        else: break
    return longest, tail

zr = np.array([longest_zero_run_and_tail(r) for r in piv.values])
customers['max_zero_run_12m'] = zr[:, 0]
customers['zero_streak_now'] = zr[:, 1]     # consecutive inactive months ending at T
customers['flight_std_12m'] = piv.std(axis=1)
customers['flight_cv_12m'] = piv.std(axis=1) / (piv.mean(axis=1) + 1)
top3 = np.sort(piv.values, axis=1)[:, -3:].sum(axis=1)
customers['top3_month_share'] = top3 / (piv.sum(axis=1).values + 1)

# --- extra momentum features (is distance / points earning slowing down?) ---
customers['flights_1m'] = agg(window(hist, 1), 'Total Flights')
d6 = agg(w6, 'Distance')
customers['dist_ratio_6v6'] = (d6 + 1) / ((customers['distance_12m'] - d6) + 1)
p6 = agg(w6, 'Points Accumulated')
customers['pts_ratio_6v6'] = (p6 + 1) / ((customers['points_acc_12m'] - p6) + 1)

# --- tenure ---
clh_idx = clh_cleaned.set_index('Loyalty Number')
enroll_ym = clh_idx['Enrollment Year'] * 12 + clh_idx['Enrollment Month']
customers['tenure_months'] = (T - enroll_ym).reindex(customers.index)

# --- demographics (one-hot; CLV & Salary deliberately excluded) ---
demo = clh_idx.reindex(customers.index)[['Loyalty Card', 'Education',
                                         'Marital Status', 'Gender',
                                         'Enrollment Type']]
demo_oh = pd.get_dummies(demo, drop_first=True, dtype=int)
customers = customers.join(demo_oh)

print(f'Feature matrix: {customers.shape[0]:,} customers x '
      f'{customers.shape[1]-1} features')
assert customers.drop(columns="churned").isnull().sum().sum() == 0, 'NaNs in features!'
print('NaN check passed: 0 missing values in feature matrix')

# ----------------------------------------------------------------------------
# PHASE 6 — MODELING
# ----------------------------------------------------------------------------
print('\n' + '=' * 70)
print('PHASE 6: MODELING')
print('=' * 70)

y = customers['churned']
X = customers.drop(columns=['churned'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RNG)
print(f'Train: {len(X_train):,} ({y_train.sum()} churners) | '
      f'Test: {len(X_test):,} ({y_test.sum()} churners)')

spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight = {spw:.1f}')

model = XGBClassifier(
    n_estimators=600, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=8,
    gamma=1, reg_lambda=2, scale_pos_weight=spw,
    eval_metric='aucpr', random_state=RNG, n_jobs=-1)

# --- 5-fold CV on the training set: honest generalization estimate +
#     out-of-fold probabilities for threshold selection (no test leakage) ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)
oof_proba = cross_val_predict(model, X_train, y_train, cv=cv,
                              method='predict_proba', n_jobs=-1)[:, 1]
cv_auc = roc_auc_score(y_train, oof_proba)
cv_pr = average_precision_score(y_train, oof_proba)
print(f'\n5-fold CV (train, out-of-fold):  ROC AUC = {cv_auc:.3f} | '
      f'PR AUC = {cv_pr:.3f}  (baseline PR = {y_train.mean():.3f})')

# --- threshold selection on OOF probabilities ---
prec, rec, thr = precision_recall_curve(y_train, oof_proba)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
f2 = 5 * prec * rec / (4 * prec + rec + 1e-9)       # recall-weighted
t_f1 = thr[np.argmax(f1[:-1])]
t_f2 = thr[np.argmax(f2[:-1])]

print('\nThreshold sweep (out-of-fold, train):')
print(f'{"thr":>6} {"precision":>10} {"recall":>8} {"F1":>7} {"flagged":>9}')
for t in [0.30, 0.40, 0.50, 0.60, t_f1, t_f2, 0.70, 0.80]:
    p = (oof_proba >= t).astype(int)
    print(f'{t:6.2f} {precision_score(y_train, p, zero_division=0):10.3f} '
          f'{recall_score(y_train, p):8.3f} '
          f'{f1_score(y_train, p, zero_division=0):7.3f} {p.sum():9,}')
print(f'\nChosen: best-F1 threshold = {t_f1:.3f} (balanced) | '
      f'best-F2 = {t_f2:.3f} (recall-leaning)')

# --- fit final model on the full training set, evaluate ONCE on test ---
model.fit(X_train, y_train)
proba_test = model.predict_proba(X_test)[:, 1]

print('\n' + '-' * 70)
print('HELD-OUT TEST SET RESULTS')
print('-' * 70)
print(f'ROC AUC = {roc_auc_score(y_test, proba_test):.3f} | '
      f'PR AUC = {average_precision_score(y_test, proba_test):.3f} '
      f'(baseline PR = {y_test.mean():.3f})')

for name, t in [('default 0.50', 0.50), (f'best-F1 {t_f1:.2f}', t_f1),
                (f'best-F2 {t_f2:.2f}', t_f2)]:
    preds = (proba_test >= t).astype(int)
    cm = confusion_matrix(y_test, preds)
    print(f'\n--- threshold: {name} ---')
    print(f'Confusion matrix:\n{cm}')
    print(classification_report(y_test, preds, digits=3,
                                target_names=['retained', 'churned']))

# --- business view: capture by risk decile (lift) ---
print('-' * 70)
print('CAPTURE BY RISK DECILE (test set) — what the marketing team acts on')
print('-' * 70)
dec = pd.DataFrame({'proba': proba_test, 'churned': y_test.values})
dec['decile'] = pd.qcut(dec['proba'].rank(method='first'), 10,
                        labels=range(10, 0, -1)).astype(int)
tab = dec.groupby('decile').agg(customers=('churned', 'size'),
                                churners=('churned', 'sum'))
tab = tab.sort_index()
tab['churn_rate'] = (tab['churners'] / tab['customers']).round(3)
tab['cum_capture'] = (tab['churners'].cumsum() / dec['churned'].sum()).round(3)
print(tab.to_string())
top10 = tab.loc[1, 'cum_capture'] if 1 in tab.index else np.nan
print(f'\nTop 10% riskiest customers capture '
      f'{tab.iloc[0]["cum_capture"]*100:.0f}% of all churners; '
      f'top 20% capture {tab.iloc[:2]["churners"].sum()/dec["churned"].sum()*100:.0f}%.')

# --- feature importance ---
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print('\nTop 15 features by importance:')
print(imp.head(15).round(4).to_string())

# ----------------------------------------------------------------------------
# PHASE 7-8 — SCORE EVERYONE, SEGMENT, ATTACH ACTIONS (deliverable file)
# ----------------------------------------------------------------------------
print('\n' + '=' * 70)
print('PHASE 7-8: SCORING + SEGMENTATION + RETENTION ACTIONS')
print('=' * 70)

# Refit on ALL labeled data for the production scorer (standard practice:
# the approach is validated above; the final scorer uses every example).
# A small 5-seed ensemble averages away XGBoost's seed-to-seed variance.
seed_probas = []
for s in range(5):
    params = model.get_params(); params['random_state'] = s
    fm = XGBClassifier(**params)
    fm.fit(X, y)
    seed_probas.append(fm.predict_proba(X)[:, 1])
final_model = fm                                   # last one kept for SHAP etc.
customers['churn_probability'] = np.mean(seed_probas, axis=0)

# Risk bands from the chosen thresholds
def risk_band(p):
    if p >= t_f1: return 'High'
    if p >= t_f2 * 0.6: return 'Medium'
    return 'Low'
customers['risk_band'] = customers['churn_probability'].apply(risk_band)

# Value axis: CLV (kept OUT of the model, used only here) + observed behavior
customers['CLV'] = clh_idx['CLV'].reindex(customers.index)
customers['loyalty_card'] = clh_idx['Loyalty Card'].reindex(customers.index)
hi_value = customers['CLV'] >= customers['CLV'].quantile(0.70)
hi_freq = customers['flights_12m'] >= customers['flights_12m'].median()

def segment(row):
    hv = row['CLV'] >= customers['CLV'].quantile(0.70)
    if row['risk_band'] == 'High' and hv:   return 'VIP at risk'
    if row['risk_band'] == 'High':          return 'At risk'
    if row['risk_band'] == 'Medium' and hv: return 'High-value, cooling'
    if row['risk_band'] == 'Medium':        return 'Cooling'
    if hv:                                  return 'High-value, healthy'
    return 'Healthy'
customers['segment'] = customers.apply(segment, axis=1)

ACTIONS = {
    'VIP at risk':         'Week 1: personal call from loyalty desk + targeted offer '
                           '(companion voucher or status extension). Owner: retention team.',
    'At risk':             'Automated win-back email within 7 days: bonus-points offer '
                           'valid 60 days, route suggestions from past trips.',
    'High-value, cooling': 'Monthly: lounge pass or seat-upgrade voucher to re-engage '
                           'before recency grows. Owner: CRM automation.',
    'Cooling':             'Quarterly re-engagement email + reminder of unspent points.',
    'High-value, healthy': 'Do not discount. Surprise-and-delight touchpoint 1x/quarter.',
    'Healthy':             'Standard newsletter cadence. No spend.',
}
customers['recommended_action'] = customers['segment'].map(ACTIONS)

print(customers.groupby('segment').agg(
    n=('churned', 'size'),
    actual_churn_rate=('churned', 'mean'),
    avg_probability=('churn_probability', 'mean'),
    avg_CLV=('CLV', 'mean')).round(3).to_string())

out = customers.reset_index()
out_path = 'final_customer_results.csv'
out.to_csv(out_path, index=False)
print(f'\nSaved {len(out):,} scored customers -> {out_path}')
print('Columns include: churn_probability, risk_band, segment, '
      'recommended_action — this file feeds the Streamlit prototype directly.')

PHASE 1-2: CLEANING
Raw cfa rows: 392,936 | raw clh rows: 16,737
Duplicate (cust,year,month) keys: 3,871 (exact copies: 1,922, conflicting: 1,949)
cfa_cleaned rows: 389,065 (expected 389,065)
Negative salaries fixed: 20 | Country column dropped (constant)
clh_cleaned rows: 16,737

PHASE 4: CHURN DEFINITION  (T = end of June 2018)
Population (active Jul17-Jun18, not cancelled by T): 13,191
Churners: 422 (3.2%)

PHASE 5: FEATURE ENGINEERING
Feature matrix: 13,191 customers x 39 features
NaN check passed: 0 missing values in feature matrix

PHASE 6: MODELING
Train: 9,893 (316 churners) | Test: 3,298 (106 churners)
scale_pos_weight = 30.3

5-fold CV (train, out-of-fold):  ROC AUC = 0.690 | PR AUC = 0.099  (baseline PR = 0.032)

Threshold sweep (out-of-fold, train):
   thr  precision   recall      F1   flagged
  0.30      0.060    0.576   0.109     3,035
  0.40      0.081    0.478   0.139     1,855
  0.50      0.105    0.408   0.168     1,224
  0.60      0.130    0.354   0.191       859
  0

In [2]:
%%writefile retention_app.py
# ============================================================================
# RETENTION CONTROL TOWER — Streamlit prototype
# ----------------------------------------------------------------------------
# Run with:   streamlit run retention_app.py
# Reads:      final_customer_results.csv  (produced by churn_pipeline.py,
#             must sit in the same folder as this file)
#
# Audience: a non-technical marketing manager. The app answers, on open:
#   1) Who needs attention right now?       (Action list tab)
#   2) How big is the problem?              (header KPIs)
#   3) What exactly do we do about it?      (Playbook tab, per segment)
#   4) Tell me about one specific member.   (Member lookup tab)
# ============================================================================

import pandas as pd
import streamlit as st

st.set_page_config(page_title="Retention Control Tower",
                   page_icon="🛫", layout="wide")

# ------------------------------------------------------------------ styling
st.markdown("""
<style>
  .block-container {padding-top: 2.2rem;}
  h1 {font-weight: 800; letter-spacing: -0.5px;}
  [data-testid="stMetricValue"] {font-size: 1.9rem;}
  .seg-card {border: 1px solid #d9d9d9; border-left: 6px solid #888;
             border-radius: 8px; padding: 14px 18px; margin-bottom: 12px;
             background: #fafafa;}
  .seg-card h4 {margin: 0 0 4px 0;}
  .seg-card .meta {color: #666; font-size: 0.85rem; margin-bottom: 6px;}
  .risk-high {border-left-color: #c62828;}
  .risk-med  {border-left-color: #ef6c00;}
  .risk-low  {border-left-color: #2e7d32;}
</style>
""", unsafe_allow_html=True)

# ------------------------------------------------------------------ data
@st.cache_data
def load_data():
    df = pd.read_csv("final_customer_results.csv")
    df["Loyalty Number"] = df["Loyalty Number"].astype(int)
    return df

try:
    df = load_data()
except FileNotFoundError:
    st.error("final_customer_results.csv not found. Run churn_pipeline.py "
             "first, then place the CSV next to this app.")
    st.stop()

SEG_RISK = {"VIP at risk": "risk-high", "At risk": "risk-high",
            "High-value, cooling": "risk-med", "Cooling": "risk-med",
            "High-value, healthy": "risk-low", "Healthy": "risk-low"}
SEG_ORDER = list(SEG_RISK.keys())

# ------------------------------------------------------------------ header
st.title("Retention Control Tower")
st.caption("Scored at the end of June 2018 · predicts disengagement risk for "
           "July–December 2018 · refresh by re-running the scoring pipeline")

at_risk = df[df["risk_band"] == "High"]
vip_risk = df[df["segment"] == "VIP at risk"]
clv_exposed = at_risk["CLV"].sum()

c1, c2, c3, c4 = st.columns(4)
c1.metric("Active members", f"{len(df):,}")
c2.metric("Flagged high risk", f"{len(at_risk):,}",
          help="Members above the model's high-risk threshold. "
               "Historically ~1 in 5 of these disengage within 6 months, "
               "vs ~1 in 30 overall.")
c3.metric("VIPs at risk", f"{len(vip_risk):,}",
          help="High risk AND top-30% lifetime value. Call these first.")
c4.metric("CLV exposed", f"${clv_exposed/1e6:,.1f}M",
          help="Combined lifetime value of all high-risk members.")

st.divider()

tab_list, tab_play, tab_member = st.tabs(
    ["📋 Action list", "🗺️ Segment playbook", "🔎 Member lookup"])

# ------------------------------------------------------------------ tab 1
with tab_list:
    st.subheader("Who needs attention first")
    st.caption("Sorted by risk. Filter, review, download — the CSV download "
               "is ready to hand to the campaign/ops team.")

    f1, f2, f3 = st.columns([2, 2, 3])
    seg_pick = f1.multiselect("Segments", SEG_ORDER,
                              default=["VIP at risk", "At risk"])
    card_pick = f2.multiselect("Loyalty card",
                               sorted(df["loyalty_card"].unique()),
                               default=sorted(df["loyalty_card"].unique()))
    top_n = f3.slider("How many members can the team contact?",
                      50, 3000, 500, step=50)

    view = df[df["segment"].isin(seg_pick) &
              df["loyalty_card"].isin(card_pick)] \
             .sort_values("churn_probability", ascending=False).head(top_n)

    if len(view) == 0:
        st.info("No members match these filters. Widen the segment or "
                "card selection above.")
    else:
        exp_churn = view["churn_probability"].sum()
        st.markdown(f"**{len(view):,} members selected** · the model expects "
                    f"roughly **{exp_churn:,.0f}** of them to disengage if "
                    f"nothing is done · combined CLV "
                    f"**${view['CLV'].sum():,.0f}**")

        show = view[["Loyalty Number", "segment", "churn_probability",
                     "CLV", "loyalty_card", "recency_months", "flights_12m",
                     "zero_streak_now", "recommended_action"]].rename(columns={
            "churn_probability": "Risk score",
            "segment": "Segment", "loyalty_card": "Card",
            "recency_months": "Months since last flight",
            "flights_12m": "Flights (12m)",
            "zero_streak_now": "Inactive streak (months)",
            "recommended_action": "Next action"})
        st.dataframe(
            show, use_container_width = True, hide_index=True, height=430,
            column_config={"Risk score": st.column_config.ProgressColumn(
                "Risk score", min_value=0, max_value=1, format="%.2f"),
                "CLV": st.column_config.NumberColumn(format="$%d")})

        st.download_button(
            "Download this contact list (CSV)",
            show.to_csv(index=False).encode(),
            file_name="retention_contact_list.csv", mime="text/csv")

# ------------------------------------------------------------------ tab 2
with tab_play:
    st.subheader("One play per segment")
    st.caption("Six segments from two questions: how likely is the member to "
               "disengage (model risk) and how valuable are they (lifetime "
               "value, top 30% = high). Each card is the standing instruction "
               "for that group.")

    stats = df.groupby("segment").agg(
        n=("Loyalty Number", "size"),
        avg_risk=("churn_probability", "mean"),
        churn_rate=("churned", "mean"),
        clv=("CLV", "mean"),
        action=("recommended_action", "first"))

    left, right = st.columns(2)
    for i, seg in enumerate(SEG_ORDER):
        if seg not in stats.index:
            continue
        r = stats.loc[seg]
        target = left if i % 2 == 0 else right
        target.markdown(f"""
<div class="seg-card {SEG_RISK[seg]}">
  <h4>{seg}</h4>
  <div class="meta">{int(r['n']):,} members · observed disengagement
  {r['churn_rate']*100:.0f}% · avg lifetime value ${r['clv']:,.0f}</div>
  <div><b>Play:</b> {r['action']}</div>
</div>""", unsafe_allow_html=True)

    st.divider()
    st.markdown("**Where the value sits**")
    pivot = df.pivot_table(index="risk_band", columns=df["CLV"] >=
                           df["CLV"].quantile(0.70),
                           values="CLV", aggfunc="sum").rename(
        columns={False: "Standard value", True: "High value (top 30%)"})
    pivot = pivot.reindex(["High", "Medium", "Low"]) / 1e6
    st.bar_chart(pivot, height=260, use_container_width= True)
    st.caption("Total lifetime value ($M) by risk band. The red bars to "
               "watch: high-value dollars sitting in High/Medium risk.")

# ------------------------------------------------------------------ tab 3
with tab_member:
    st.subheader("Look up one member")
    q = st.text_input("Loyalty number", placeholder="e.g. 100018")
    if q:
        try:
            row = df[df["Loyalty Number"] == int(q)]
        except ValueError:
            row = df.iloc[0:0]
        if len(row) == 0:
            st.warning("No member with that loyalty number is in the scored "
                       "population (members with no flights in the last 12 "
                       "months are out of scope).")
        else:
            r = row.iloc[0]
            a, b, c, d = st.columns(4)
            a.metric("Risk score", f"{r['churn_probability']:.2f}")
            b.metric("Segment", r["segment"])
            c.metric("Lifetime value", f"${r['CLV']:,.0f}")
            d.metric("Card", r["loyalty_card"])
            e, f, g, h = st.columns(4)
            e.metric("Flights, last 12m", int(r["flights_12m"]))
            f.metric("Months since last flight", int(r["recency_months"]))
            g.metric("Inactive streak now", f"{int(r['zero_streak_now'])} mo")
            h.metric("Tenure", f"{int(r['tenure_months'])} mo")
            st.markdown(f"**Recommended next action:** "
                        f"{r['recommended_action']}")

Overwriting retention_app.py
